In [ ]:
import torch


def ellipse_perimeter(
    a,
    b,
    pi=torch.pi,
    one=torch.scalar_tensor(1),
    three=torch.scalar_tensor(3),
    ten=torch.scalar_tensor(10),
    four=torch.scalar_tensor(4),
):
    h = ((a - b) ** 2) / ((a + b) ** 2)
    return (
        pi
        * (a + b)
        * (one + (three * h) / (ten + torch.sqrt(torch.relu(four - three * h))))
    )


def compute_loss(approximator, a, b):
    args = tuple(torch.cat((torch.stack((a, b)), approximator[0])))
    return (ellipse_perimeter(a, b) - ellipse_perimeter(*args)) ** 2


ellipse_approximator = torch.Tensor([3, 1, 3, 10, 4]).unsqueeze(0)

In [ ]:
sample_size = 1000
epochs = 100

ellipses = torch.stack(
    [
        torch.Tensor([a, b])
        for (a, b) in list(
            set(
                [
                    (x + 1, y * y // (x + 1) + 1)
                    for (x, y) in zip(range(sample_size), range(sample_size))
                ]
                + [(1, x + 1) for x in range(sample_size)]
            )
        )
    ]
    * epochs
)

In [ ]:
from tqdm import tqdm


def train(ea, s):
    ea = ea.clone().float().requires_grad_(True)

    optimizer = torch.optim.Adam([ea], lr=1e-4)
    pbar = tqdm(s, desc="Training")

    for t in pbar:
        optimizer.zero_grad()

        loss = compute_loss(ea, t[0], t[1])

        loss.backward()
        optimizer.step()

        pbar.set_postfix(mse=loss.item())

    return ea


train(ellipse_approximator.to("cuda"), ellipses.to("cuda"))